# 03 · LangGraph temporal-routing agent

Wraps the already-verified temporal-routing retrieval pipeline (`tau.retrieval.router.retrieve_with_temporal_routing`) in a small [LangGraph](https://langchain-ai.github.io/langgraph/) graph, `tau.agent.graph.build_temporal_agent_graph`.

This notebook does **not** re-implement retrieval. Every graph node is a thin wrapper around functions already used and verified in [02_retrieval_baseline.ipynb](02_retrieval_baseline.ipynb) — `classify_query`, `extract_time_window`, `semantic_search`, `rerank_results`, `apply_temporal_decay`. LangGraph only sequences them.

Assumes notebook 02 has already been run at least once, so Postgres' `documents` table is populated with embeddings.

**Kernel must be `Python (tau)`** (this project's `.venv`) — the first cell prints the interpreter in use so you can confirm before going further.

## 1. Setup

In [1]:
import sys
from pathlib import Path

print("PYTHON:", sys.executable)

import tau
print("tau package:", Path(tau.__file__).parent)


PYTHON: /Users/akshay/tau/.venv/bin/python
tau package: /Users/akshay/tau/src/tau


In [2]:
import os
from dotenv import load_dotenv

project_root = Path.cwd().parent
load_dotenv(project_root / ".env")

VOYAGE_API_KEY = os.environ["VOYAGE_API_KEY"]
PG_DSN = os.environ.get("PG_DSN", "dbname=tau")

print("VOYAGE_API_KEY loaded:", bool(VOYAGE_API_KEY))
print("PG_DSN:", PG_DSN)


VOYAGE_API_KEY loaded: True
PG_DSN: dbname=tau


In [3]:
import psycopg
from pgvector.psycopg import register_vector

pg_conn = psycopg.connect(PG_DSN, autocommit=True)
register_vector(pg_conn)

row_count = pg_conn.execute("SELECT COUNT(*) FROM documents").fetchone()[0]
print("Connected to Postgres:", pg_conn.info.dbname)
print("documents row count:", row_count)

assert row_count > 0, (
    "documents table is empty — run notebook 02 first to embed and load documents"
)


Connected to Postgres: tau
documents row count: 114


## 2. What LangGraph is doing here

`build_temporal_agent_graph` (in [src/tau/agent/graph.py](../src/tau/agent/graph.py)) compiles a 5-node graph:

```
START -> classify_intent -> retrieve -> rerank -> [apply_tau | finalize] -> END
```

- **classify_intent** — calls `classify_query` + (if `explicit_temporal`) `extract_time_window`. Sets `route` and `time_window`.
- **retrieve** — calls `embed_query` + `semantic_search`. The `time_window` from state decides whether `semantic_search` hard-filters by `published_at` (`explicit_temporal`) or searches the whole corpus (`current` / `topical`). This is where "choose retrieval path" actually happens — it's the same `time_window`-driven branch already used by `retrieve_with_temporal_routing`.
- **rerank** — calls `rerank_results`.
- **apply_tau** — calls `apply_temporal_decay`. Only reached when `route == "current"` (a conditional edge after `rerank`).
- **finalize** — for `explicit_temporal` / `topical`, the reranked order is already final; no decay applies.

No planner/critic/researcher/writer/supervisor agents, no MCP, no memory — one state dict flowing through five small, single-purpose nodes. `pg_conn` and `VOYAGE_API_KEY` are captured as closures when the graph is built, not stored in the state — the state only holds the data that actually flows through the pipeline: `query`, `route`, `time_window`, `results`, `reranked_results`, `tau_results`, `final_results`.

## 3. Build/import the graph

In [4]:
from tau.agent.graph import build_temporal_agent_graph

graph = build_temporal_agent_graph(
    pg_conn,
    VOYAGE_API_KEY,
    limit=20,
    top_k=5,
    tau_hours=24,
)

compiled_graph = graph.get_graph()
print("nodes:", list(compiled_graph.nodes))
print()
print("edges:")
for edge in compiled_graph.edges:
    label = f'  [{edge.conditional}]' if edge.conditional else ''
    print(f'  {edge.source} -> {edge.target}{label}')


nodes: ['__start__', 'classify_intent', 'retrieve', 'rerank', 'apply_tau', 'finalize', '__end__']

edges:
  __start__ -> classify_intent
  classify_intent -> retrieve
  rerank -> apply_tau  [True]
  rerank -> finalize  [True]
  retrieve -> rerank
  apply_tau -> __end__
  finalize -> __end__


## 4. Test explicit temporal query

Expected: `route=explicit_temporal`, a `time_window` is extracted, no tau decay (`final_results` has no `recency_weight` / `final_score`).

In [5]:
explicit_query = "What happened with OpenAI in the last 3 hours?"
explicit_state = graph.invoke({"query": explicit_query})

print(f'query={explicit_state["query"]!r}')
print(f'route={explicit_state["route"]}')
print(f'time_window={explicit_state["time_window"]}')
print('retrieval path: hard time-window filter -> semantic search -> rerank -> no tau decay')
print()

for r in explicit_state["final_results"]:
    print(f'  - title={r["title"]!r}')
    print(f'    source={r["source"]}  published_at={r["published_at"]}')
    print(f'    rerank_score={r["rerank_score"]:.4f}')


query='What happened with OpenAI in the last 3 hours?'
route=explicit_temporal
time_window=(datetime.datetime(2026, 8, 18, 22, 36, 32, 6418, tzinfo=datetime.timezone.utc), datetime.datetime(2026, 8, 19, 1, 36, 32, 6418, tzinfo=datetime.timezone.utc))
retrieval path: hard time-window filter -> semantic search -> rerank -> no tau decay

  - title='OpenAI lays out new security changes after its AI hacked Hugging Face'
    source=hackernews  published_at=2026-08-18 20:07:06-04:00
    rerank_score=0.6797
  - title="OpenAI's overhead will rise 20 percent for some workloads as it hardens security"
    source=hackernews  published_at=2026-08-18 20:14:58-04:00
    rerank_score=0.4805
  - title='Could End the RAM Crisis [video]'
    source=hackernews  published_at=2026-08-18 20:15:19-04:00
    rerank_score=0.3320
  - title='Cerebras CS-4 rack systems juice chips for every last drop of AI performance'
    source=hackernews  published_at=2026-08-18 20:14:25-04:00
    rerank_score=0.3262
  - title=

## 5. Test current query

Expected: `route=current`, no `time_window`, tau decay applied (`final_results` has `rerank_score`, `recency_weight`, `final_score`).

In [6]:
current_query = "What's going on with OpenAI?"
current_state = graph.invoke({"query": current_query})

print(f'query={current_state["query"]!r}')
print(f'route={current_state["route"]}')
print(f'time_window={current_state["time_window"]}')
print('retrieval path: semantic search -> rerank -> tau decay (tau_hours=24)')
print()

for r in current_state["final_results"]:
    print(f'  - title={r["title"]!r}')
    print(f'    source={r["source"]}  published_at={r["published_at"]}')
    print(
        f'    rerank_score={r["rerank_score"]:.4f}  '
        f'recency_weight={r["recency_weight"]:.4f}  '
        f'final_score={r["final_score"]:.4f}'
    )


query="What's going on with OpenAI?"
route=current
time_window=None
retrieval path: semantic search -> rerank -> tau decay (tau_hours=24)

  - title='OpenAI lays out new security changes after its AI hacked Hugging Face'
    source=hackernews  published_at=2026-08-18 20:07:06-04:00
    rerank_score=0.7422  recency_weight=0.9398  final_score=0.6975
  - title="OpenAI's overhead will rise 20 percent for some workloads as it hardens security"
    source=hackernews  published_at=2026-08-18 20:14:58-04:00
    rerank_score=0.5977  recency_weight=0.9449  final_score=0.5647
  - title='OpenAI Overhauls Safety Protocols After Its AI Agents Went Rogue'
    source=wired  published_at=2026-08-18 14:33:11-04:00
    rerank_score=0.7266  recency_weight=0.7453  final_score=0.5415
  - title='The Safety Reckoning Inside OpenAI'
    source=wired  published_at=2026-08-13 18:37:19-04:00
    rerank_score=0.6719  recency_weight=0.0059  final_score=0.0040
  - title='The White House Is Going to Expand Its AI Pol

## 6. Test topical query

Expected: `route=topical`, no `time_window`, no tau decay (`final_results` has no `recency_weight` / `final_score`).

In [7]:
topical_query = "Explain OpenAI's safety approach"
topical_state = graph.invoke({"query": topical_query})

print(f'query={topical_state["query"]!r}')
print(f'route={topical_state["route"]}')
print(f'time_window={topical_state["time_window"]}')
print('retrieval path: semantic search -> rerank -> no tau decay')
print()

for r in topical_state["final_results"]:
    print(f'  - title={r["title"]!r}')
    print(f'    source={r["source"]}  published_at={r["published_at"]}')
    print(f'    rerank_score={r["rerank_score"]:.4f}')


query="Explain OpenAI's safety approach"
route=topical
time_window=None
retrieval path: semantic search -> rerank -> no tau decay

  - title='OpenAI Overhauls Safety Protocols After Its AI Agents Went Rogue'
    source=wired  published_at=2026-08-18 14:33:11-04:00
    rerank_score=0.6875
  - title='The Safety Reckoning Inside OpenAI'
    source=wired  published_at=2026-08-13 18:37:19-04:00
    rerank_score=0.6250
  - title='OpenAI lays out new security changes after its AI hacked Hugging Face'
    source=hackernews  published_at=2026-08-18 20:07:06-04:00
    rerank_score=0.4980
  - title="OpenAI's overhead will rise 20 percent for some workloads as it hardens security"
    source=hackernews  published_at=2026-08-18 20:14:58-04:00
    rerank_score=0.4590
  - title='The White House Is Going to Expand Its AI Policy'
    source=wired  published_at=2026-08-12 17:00:00-04:00
    rerank_score=0.3711


## 7. Inspect state transitions/results

Each route walks a different subset of the graph, which shows up directly in which state keys got populated. `tau_results` only exists when `apply_tau` actually ran (the `current` route) — `finalize` never sets it.

In [8]:
routing_states = {
    "explicit_temporal": explicit_state,
    "current": current_state,
    "topical": topical_state,
}

for route_name, state in routing_states.items():
    print(f'{route_name}: state keys = {sorted(state.keys())}')


explicit_temporal: state keys = ['final_results', 'query', 'reranked_results', 'results', 'route', 'time_window']
current: state keys = ['final_results', 'query', 'reranked_results', 'results', 'route', 'tau_results', 'time_window']
topical: state keys = ['final_results', 'query', 'reranked_results', 'results', 'route', 'time_window']


In [9]:
# --- Assertion 1: explicit_temporal does not apply tau ---------------------
assert "tau_results" not in explicit_state, "explicit_temporal should not run apply_tau"
for r in explicit_state["final_results"]:
    assert "recency_weight" not in r and "final_score" not in r, (
        f'{r["title"]!r} unexpectedly carries tau-decay fields for explicit_temporal'
    )
print("PASS: explicit_temporal — no tau decay applied")

# --- Assertion 2: current does apply tau ------------------------------------
assert "tau_results" in current_state, "current should run apply_tau"
for r in current_state["final_results"]:
    assert "recency_weight" in r and "final_score" in r, (
        f'{r["title"]!r} is missing tau-decay fields for the current route'
    )
print("PASS: current — tau decay applied")

# --- Assertion 3: topical does not apply tau --------------------------------
assert "tau_results" not in topical_state, "topical should not run apply_tau"
for r in topical_state["final_results"]:
    assert "recency_weight" not in r and "final_score" not in r, (
        f'{r["title"]!r} unexpectedly carries tau-decay fields for topical'
    )
print("PASS: topical — no tau decay applied")

# --- Assertion 4: explicit_temporal results respect the time window --------
start_time, end_time = explicit_state["time_window"]
assert explicit_state["final_results"], "explicit_temporal returned no results to check"
for r in explicit_state["final_results"]:
    assert r["published_at"] is not None, f'{r["title"]!r} has no published_at'
    assert start_time <= r["published_at"] <= end_time, (
        f'{r["title"]!r} published_at={r["published_at"]} outside window '
        f'[{start_time}, {end_time}]'
    )
print("PASS: explicit_temporal — every published_at is inside the extracted window")

print()
print("All routing assertions passed.")


PASS: explicit_temporal — no tau decay applied
PASS: current — tau decay applied
PASS: topical — no tau decay applied
PASS: explicit_temporal — every published_at is inside the extracted window

All routing assertions passed.


## 8. Summary

`tau.agent.graph.build_temporal_agent_graph` reproduces `retrieve_with_temporal_routing`'s three routes as an explicit, inspectable LangGraph graph — 5 nodes, one conditional edge, no new retrieval logic:

| route | time_window | retrieval | tau decay |
|---|---|---|---|
| `explicit_temporal` | extracted | hard-filtered semantic search -> rerank | no |
| `current` | none | semantic search -> rerank | yes (`tau_hours=24`) |
| `topical` | none | semantic search -> rerank | no |

All three routes were exercised above with the graph (not by calling `retrieve_with_temporal_routing` directly), and all four verification assertions passed. `tau.retrieval.*` remains the single source of truth for retrieval behavior — this graph only orchestrates it.

**Stopping here per scope** — no MCP, no multi-agent setup, no memory, no eval harness. Those are explicitly out of scope for this notebook.